In [ ]:
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis

In [ ]:
velocity = na.linspace(-250, 250, axis="wavelength", num=21) * u.km / u.s

In [ ]:
wavelength_rest = 630 * u.AA

In [ ]:
position_scene = na.Cartesian2dVectorLinearSpace(
    start=-10 * u.arcsec,
    stop=+10 * u.arcsec,
    axis=na.Cartesian2dVectorArray("scene_x", "scene_y"),
    num=33,
)

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 97, axis="sensor_x") * u.pix,
    y=na.arange(0, 97, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_scene = na.DopplerPositionalVectorArray.from_velocity(
    velocity=velocity,
    wavelength_rest=wavelength_rest,
    position=position_scene,
)

In [ ]:
coordinates_sensor = na.DopplerPositionalVectorArray.from_velocity(
    velocity=velocity,
    wavelength_rest=wavelength_rest,
    position=position_sensor,
)

In [ ]:
scene = ctis.scenes.gaussians(coordinates_scene)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[0.9, 0.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    colorbar = na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"scene $y$ ({ax.get_ylabel()})")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
angle = na.linspace(0, 360, num=4, axis="channel", endpoint=False) * u.deg

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=1 * u.cm**2,
    timedelta_exposure=20 * u.s,
    plate_scale=0.625 * u.arcsec / u.pix,
    dispersion=0.021 * u.AA / u.pix,
    angle=angle,
    wavelength_ref=wavelength_rest,
    position_ref=48 * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    channel="dispersion angle = " + angle.to_string_array("%03d"),
    axis_channel="channel",
    axis_wavelength="wavelength",
    axis_scene_xy=("scene_x", "scene_y"),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
images = instrument.image(scene)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=instrument.num_channel,
        figsize=(11, 3.2),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    na.plt.pcolormesh(
        images.inputs.position.x,
        images.inputs.position.y,
        C=images.outputs.value,
        ax=na.ScalarArray(axs, axes=("channel",)),
        cmap="gray",
    )
    for ax, name in zip(axs, instrument.channel.ndarray):
        ax.set_aspect("equal")
        ax.set_title(name, fontsize=9)

In [ ]:
model = ctis.inverters.GaussianModel(
    width_thermal=11 * u.km / u.s,
    width_instrument=8 * u.km / u.s,
    velocity_max=250 * u.km / u.s,
)

In [ ]:
inverter = ctis.inverters.ParametricInverter(
    instrument=instrument,
    model=model,
    num_iteration=1000,
)

In [ ]:
inversion = inverter(images)

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(
        inversion.iteration,
        inversion.mean_chi_squared,
        ax=ax,
    )
    ax.set_yscale("log")
    ax.set_xlabel("iteration")
    ax.set_ylabel(r"$\langle \chi^2 \rangle$")

In [ ]:
solution = inversion.solution

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        figsize=(9, 4),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    for ax, cube, title in zip(axs, [scene, solution], ["truth", "reconstructed"]):
        na.plt.rgbmesh(
            C=cube,
            axis_wavelength="wavelength",
            ax=ax,
            vmin=0,
            vmax=scene.outputs.max(),
        )
        ax.set_aspect("equal")
        ax.set_title(title)
        ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    axs[0].set_ylabel(f"scene $y$ ({axs[0].get_ylabel()})")

In [ ]:
parameters = inversion.parameters
list(parameters)

In [ ]:
keys = ["intensity", "velocity", "width_nonthermal"]
cmaps = ["inferno", "RdBu_r", "viridis"]

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        figsize=(12, 3.6),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    for ax, key, cmap in zip(axs, keys, cmaps):
        parameter = parameters[key]
        img = na.plt.pcolormesh(
            coordinates_scene.position.x,
            coordinates_scene.position.y,
            C=parameter.value,
            ax=ax,
            cmap=cmap,
        )
        ax.set_aspect("equal")
        ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
        plt.colorbar(
            img.ndarray.item(),
            ax=ax,
            label=f"{key} ({parameter.unit:latex_inline})",
        )
    axs[0].set_ylabel(f"scene $y$ ({axs[0].get_ylabel()})")

In [ ]:
velocity_true = na.pdf.median(
    x=scene.inputs.velocity,
    f=scene.outputs,
    axis="wavelength",
)

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        figsize=(9, 3.6),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    for ax, C, title in zip(
        axs,
        [velocity_true, parameters["velocity"]],
        ["true median velocity", "fitted velocity"],
    ):
        img = na.plt.pcolormesh(
            coordinates_scene.position.x,
            coordinates_scene.position.y,
            C=C.value,
            ax=ax,
            cmap="RdBu_r",
            vmin=-200,
            vmax=+200,
        )
        ax.set_aspect("equal")
        ax.set_title(title)
        ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    axs[0].set_ylabel(f"scene $y$ ({axs[0].get_ylabel()})")
    plt.colorbar(img.ndarray.item(), ax=axs, label="velocity (km / s)")

In [ ]:
inversion.plot_moments(scene, axis="wavelength");